In [3]:
# ruff: noqa: T201

import contextlib
import pathlib
import time

import pandas as pd
from typing_extensions import Final

import svglab


SOURCE_DIR: Final = pathlib.Path("~/Downloads/freesvg").expanduser()
MIN_SIZE: Final = 10 * 1000  # 10 KB
MAX_SIZE: Final = 50 * 1000  # 50 KB
SAMPLES: Final = 1000
SEED: Final = 42
OUTPUT: Final = pathlib.Path("data.csv")


data = pd.DataFrame(
    (
        (path.name, path.stat().st_size)
        for path in SOURCE_DIR.glob("*.svg")
    ),
    columns=["filename", "size"],
)

with_good_size = data[
    (data["size"] > MIN_SIZE) & (data["size"] < MAX_SIZE)
]

sampled = with_good_size.sample(n=SAMPLES, random_state=SEED)

for filename, _ in sampled.itertuples(index=False):
    timer = time.perf_counter()

    with (
        (SOURCE_DIR / filename).open("r") as f,
        contextlib.suppress(Exception),
    ):
        svglab.parse_svg(f)

    elapsed = (time.perf_counter() - timer) * 1000

    sampled.loc[sampled["filename"] == filename, "time"] = elapsed

sampled.to_csv(OUTPUT, index=False)
